In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. **Install Ollama** from [https://ollama.com/download](https://ollama.com/download) for your operating system.
   - **macOS**: download the `.pkg` and install it.
   - **Windows**: download the `.msi` and install it.
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally** by opening a terminal and running:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. **Optional: test the local server** with:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}  
   ```

4. **Optional: use it from Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": your_prompt}]
   )

   print(response['message']['content'])
   ```


In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The context doesn’t mention **Olama** specifically.

If you mean **running the course locally**, the FAQ says you can do that instead of using Codespaces, as long as you’re comfortable setting up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

It also says that if you run locally, you should:

- document your setup
- keep your environment reproducible

If you meant a specific local setup for an LLM tool, I’d need the relevant course FAQ entry.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether registration is still open.\n\nUsually, you can join if:\n- the course hasn’t started yet, or\n- it allows late enrollment / rolling admission, or\n- there are still seats available.\n\nTo find out quickly:\n1. Check the course page for enrollment dates or a “join” button.\n2. Contact the instructor or course support.\n3. If you want, send me the course name or the page text, and I can help you figure out whether you can still enroll.'

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join the course"}', call_id='call_GpYnmlca3bY8qshz9pNDFnH2', name='search', type='function_call', id='fc_0dd440a7c6b87c24006a37bae2eddc81a3aa7c68dd51bca9db', namespace=None, status='completed')]

In [12]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join the course"}', call_id='call_GpYnmlca3bY8qshz9pNDFnH2', name='search', type='function_call', id='fc_0dd440a7c6b87c24006a37bae2eddc81a3aa7c68dd51bca9db', namespace=None, status='completed')

In [13]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered late enrollment can I join the course'}

In [14]:
call.name

'search'

In [15]:
results = search(**args)
results 

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I sta

In [16]:
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course

In [17]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [18]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [19]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join the course"}', call_id='call_GpYnmlca3bY8qshz9pNDFnH2', name='search', type='function_call', id='fc_0dd440a7c6b87c24006a37bae2eddc81a3aa7c68dd51bca9db', namespace=None, status='completed')

In [20]:
messages.append(call)

In [21]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join the course"}', call_id='call_GpYnmlca3bY8qshz9pNDFnH2', name='search', type='function_call', id='fc_0dd440a7c6b87c24006a37bae2eddc81a3aa7c68dd51bca9db', namespace=None, status='completed')]

In [22]:
# messages.extend(response.output)

# messages.append({
#  "type": "function_call_output",
#  "call_id": call.call_id,
#  "output": result_json,
# })

In [23]:
messages.append(function_call_output)

In [24]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join the course"}', call_id='call_GpYnmlca3bY8qshz9pNDFnH2', name='search', type='function_call', id='fc_0dd440a7c6b87c24006a37bae2eddc81a3aa7c68dd51bca9db', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_GpYnmlca3bY8qshz9pNDFnH2',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate

In [25]:
response

Response(id='resp_0dd440a7c6b87c24006a37bae26fe481a3a3cf048b45348db1', created_at=1782037218.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join the course"}', call_id='call_GpYnmlca3bY8qshz9pNDFnH2', name='search', type='function_call', id='fc_0dd440a7c6b87c24006a37bae2eddc81a3aa7c68dd51bca9db', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ database for entries matching the given query.')], top_p=0.98, background=False, completed_at=1782037219.0, conversa

In [26]:
response.usage

ResponseUsage(input_tokens=69, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=26, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=95)

In [27]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(69, 26)

In [28]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [29]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join the course"}', call_id='call_GpYnmlca3bY8qshz9pNDFnH2', name='search', type='function_call', id='fc_0dd440a7c6b87c24006a37bae2eddc81a3aa7c68dd51bca9db', namespace=None, status='completed')

In [30]:
call.arguments

'{"query":"join course discovered late enrollment can I join the course"}'

In [31]:
search(**json.loads(call.arguments))

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I sta

In [32]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [33]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [34]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [35]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [36]:
response

Response(id='resp_0f1d11b093ad19c4006a37bae3ee28819f92fd68210c941a9d', created_at=1782037219.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"join course enroll registration discovered course can I join"}', call_id='call_JDCYYIxMcffDSTHHN3o6RM36', name='search', type='function_call', id='fc_0f1d11b093ad19c4006a37bae48884819f95fa162f8fd84761', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course enrollment late join sign up FAQ"}', call_id='call_mrS07GPeISL5wjcON6e7gp1I', name='search', type='function_call', id='fc_0f1d11b093ad19c4006a37bae48898819f904e29e2a1c6ddde', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the 

In [37]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [38]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course enroll registration discovered course can I join"}', call_id='call_JDCYYIxMcffDSTHHN3o6RM36', name='search', type='function_call', id='fc_0f1d11b093ad19c4006a37bae48884819f95fa162f8fd84761', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment late join sign up FAQ"}', call_id='call_mrS07GPeISL5wjcON6e7gp1I', name='search', type='function_call', id='fc_0f1d11b093ad19c4006a37bae48898819f904e29e2a1c6ddde', namespace=None, status='completed')]

In [39]:
for item in response.output:
    print(item)
    print(item.name)
    print(item.arguments)
    print("===========")
    print(make_call(item))

ResponseFunctionToolCall(arguments='{"query":"join course enroll registration discovered course can I join"}', call_id='call_JDCYYIxMcffDSTHHN3o6RM36', name='search', type='function_call', id='fc_0f1d11b093ad19c4006a37bae48884819f95fa162f8fd84761', namespace=None, status='completed')
search
{"query":"join course enroll registration discovered course can I join"}
{'type': 'function_call_output', 'call_id': 'call_JDCYYIxMcffDSTHHN3o6RM36', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n

In [40]:
make_call

<function __main__.make_call(call)>

In [41]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course enroll registration discovered course can I join"}
function_call: search {"query":"course enrollment late join sign up FAQ"}


In [42]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enroll registration discovered course can I join"}', call_id='call_JDCYYIxMcffDSTHHN3o6RM36', name='search', type='function_call', id='fc_0f1d11b093ad19c4006a37bae48884819f95fa162f8fd84761', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment late join sign up FAQ"}', call_id='call_mrS07GPeISL5wjcON6e7g

In [43]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...


function_call: search {"query":"join course discovered can I join enrollment registration late join course"}
function_call: search {"query":"course FAQ join discovered can I join now enrollment"}
function_call: search {"query":"students can join the course after it starts enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. If you’re only aiming to learn, you can start anytime with the course materials.

If you want, I can also help you figure out:
- how to start the course,
- whether you can still get a certificate,
- or what to do next as a late joiner.


In [44]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [45]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...


function_call: search {"query":"Olama locally run install local model Ollama course FAQ"}
function_call: search {"query":"run Ollama locally install use terminal local model FAQ"}
function_call: search {"query":"Ollama local setup macOS Windows Linux FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**, do this:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model=

'To run **Ollama locally**, do this:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\

In [46]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [47]:
result = agent_loop(instructions, question)

iteration #1...


function_call: search {"query":"join course late enrollment discovered course can I join"}
iteration #2...
function_call: search {"query":"certificate project submit while accepting submissions live cohort self-paced join course discovered course"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you’ll need to submit your project while submissions are still open. The course can also be followed in a self-paced way, but certificates are only awarded when you finish with a live cohort.

If you want, I can also help you figure out how to start from where you are now.


In [48]:
result

'Yes — you can still join the course.\n\nIf you want a certificate, though, you’ll need to submit your project while submissions are still open. The course can also be followed in a self-paced way, but certificates are only awarded when you finish with a live cohort.\n\nIf you want, I can also help you figure out how to start from where you are now.'

In [49]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...


function_call: search {"query":"queen gambit what is queen's gambit course FAQ"}
iteration #2...
function_call: search {"query":"queen gambit queen's gambit course FAQ chess"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit,” and it looks off-topic for the course.

If you meant something course-related, please rephrase and I’ll check the FAQ again. Are there other areas you want to explore?


In [50]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [51]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [52]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [53]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [54]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [55]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [56]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [57]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


In [58]:
result.cost

CostInfo(input_cost=Decimal('0.0010485'), output_cost=Decimal('0.0011565'), total_cost=Decimal('0.0022050'))

In [59]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local Ollama how do I run locally"}', c

In [60]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
# runner.run();


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


...

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received
